# Exercise D — Build a CMIP6 Data Pipeline (with AI as co-pilot)
**LEAP Climate Data Science Bootcamp · Summer 2026 - Ayon Roy**

## Goal

Use an AI assistant to build a **working four-stage climate-projection pipeline**:

| Stage | Verb | What it does |
|---|---|---|
| 1 | **Acquire**   | Pull a CMIP6 `tas` variable for two SSP scenarios |
| 2 | **Process**   | Regrid to a common 1°×1° grid; mask to a region |
| 3 | **Analyze**   | Compute decadal mean warming and multi-model spread |
| 4 | **Visualize** | Cartopy map of regional warming with model-agreement stippling |

## The prompting strategy that actually works

1. **One stage at a time.** Don't ask the AI for the whole pipeline in one message — you'll
   get plausible-looking but broken code. Prompt for stage 1, run it, verify, *then* prompt for stage 2.
2. **Pin your environment.** Tell the AI which library versions you have. *"Using xarray 2024,
   xesmf 0.8, intake-esm 2024 — what's the right syntax for conservative regridding?"*
3. **Run and report back.** After running each generated cell, paste the actual output
   (or traceback) into the chat **before** asking for the next step. Don't let the AI hallucinate the result.
4. **Question the silence.** If the AI confidently produces 30 lines with no warnings, *ask* it:
   "What could go wrong with this code? What edge cases are you not handling?" — it often catches its own bugs.

## What's pre-built for you

- Imports for `xarray`, `intake-esm`, `xesmf`, `cartopy`, `matplotlib`.
- The Pangeo CMIP6 cloud catalog is loaded and ready — just pick variables.
- Empty cells labeled **TODO Stage N** for each pipeline stage.
- A **reference "good" figure** at the bottom so you know what you're aiming for.
- A fallback small NetCDF sample if you don't have internet (or Pangeo is being moody).

## What you build

Pick a **region of interest** (e.g., Western Europe, the Sahel, the Andes, Bangladesh)
and compute the **late-21st-century warming** in that region under two SSP scenarios.
The deliverable is a publication-style figure with a brief caption.

## 1. Setup — imports and environment

In [ ]:
# On Colab, uncomment the next line. xesmf can be slow to install — give it ~2 minutes.
# !pip install -q xarray intake-esm xesmf cartopy gcsfs zarr matplotlib pooch

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# These two are optional — the notebook degrades gracefully if they're not installed.
try:
    import intake
    HAVE_INTAKE = True
except ImportError:
    HAVE_INTAKE = False
    print('intake not installed — will use the offline fallback dataset.')

try:
    import xesmf as xe
    HAVE_XESMF = True
except ImportError:
    HAVE_XESMF = False
    print('xesmf not installed — regridding will use a simple interp() fallback.')

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False
    print('cartopy not installed — will plot with plain matplotlib.')

print(f'xarray   {xr.__version__}')
print(f'have intake-esm: {HAVE_INTAKE}')
print(f'have xesmf:      {HAVE_XESMF}')
print(f'have cartopy:    {HAVE_CARTOPY}')

## 2. Configuration — pick your region

Edit the box below to choose your region. The defaults are **Western Europe**
(35°N–60°N, 10°W–25°E). Try the Sahel (10°N–20°N, 20°W–40°E) or Bangladesh
(20°N–27°N, 88°E–93°E) or somewhere you actually care about.

In [ ]:
# CONFIG
REGION = {
    'name': 'Western Europe',
    'lat_min': 35, 'lat_max': 60,
    'lon_min': -10, 'lon_max': 25,
}

VARIABLE = 'tas'                # near-surface air temperature
SCENARIOS = ['ssp245', 'ssp585']  # moderate vs high emissions
HISTORICAL_REF = ('1995', '2014') # 20-yr historical reference (matches IPCC AR6)
FUTURE_PERIOD  = ('2081', '2100') # late-21st-century

TARGET_RES = 1.0  # degrees, for the common regrid target

print(f'Region: {REGION["name"]}  '
      f'({REGION["lat_min"]}–{REGION["lat_max"]} N, '
      f'{REGION["lon_min"]}–{REGION["lon_max"]} E)')
print(f'Variable : {VARIABLE}')
print(f'Scenarios: {SCENARIOS}')
print(f'Reference: {HISTORICAL_REF[0]}–{HISTORICAL_REF[1]}')
print(f'Future   : {FUTURE_PERIOD[0]}–{FUTURE_PERIOD[1]}')

## Stage 1 — Acquire

**Your task:** pull `tas` for **3–5 CMIP6 models** for both `ssp245` and `ssp585`,
plus the matching `historical` runs. Use the **Pangeo CMIP6 cloud catalog** if you
have internet; the fallback below loads a tiny offline sample if you don't.

**Suggested prompt to your AI tool:**

> *I'm using intake-esm 2024 to query the Pangeo CMIP6 catalog at*
> *https://storage.googleapis.com/cmip6/pangeo-cmip6.json. Write a query that*
> *returns monthly `tas` for the `historical`, `ssp245`, and `ssp585` experiments*
> *for 3 well-known models (e.g. MPI-ESM1-2-LR, CanESM5, MIROC6), member r1i1p1f1,*
> *table_id Amon. Then open the matching zarr stores as a dict of xarray Datasets.*
> *Use intake-esm's `to_dataset_dict()` and show me how to handle the case where*
> *some models are missing one of the scenarios.*

Run, verify the shapes, **paste the output back** to the AI before moving on.

### Offline fallback

If Pangeo is unreachable, the cell below builds a small synthetic CMIP6-style
Dataset (3 'models' × 3 experiments × ~100 years monthly) so you can still develop
the pipeline. Real Pangeo data is much bigger but the *shape* is the same.

In [ ]:
# ------------------------------------------------------------
# OFFLINE FALLBACK — synthetic CMIP6-style data
# Use this if Pangeo is unreachable or you're working offline.
# ------------------------------------------------------------
def make_synthetic_cmip6():
    """Generate a tiny CMIP6-style dict-of-Datasets for offline use."""
    rng = np.random.default_rng(0)
    lat = np.arange(-89, 90, 2.0)
    lon = np.arange(0, 360, 2.0)
    out = {}
    for model in ['SYN-MPI', 'SYN-CAN', 'SYN-MIR']:
        # model-specific climate sensitivity
        cs = rng.uniform(2.5, 4.5)
        for exp in ['historical', 'ssp245', 'ssp585']:
            if exp == 'historical':
                time = pd.date_range('1850-01-01', '2014-12-31', freq='MS')
                trend = 0.01 * cs * (time.year - 1850) / 100.0
            elif exp == 'ssp245':
                time = pd.date_range('2015-01-01', '2100-12-31', freq='MS')
                trend = 0.01 * cs * (time.year - 1850) / 100.0 + 0.015 * cs * (time.year - 2015) / 100.0
            else:  # ssp585
                time = pd.date_range('2015-01-01', '2100-12-31', freq='MS')
                trend = 0.01 * cs * (time.year - 1850) / 100.0 + 0.05 * cs * (time.year - 2015) / 100.0

            base = 273.15 + 15.0 - 0.5 * np.abs(lat)[None, :, None]
            base = np.broadcast_to(base, (len(time), len(lat), len(lon)))
            field = base + trend.values[:, None, None] + rng.normal(0, 0.3, base.shape).astype('float32')
            ds = xr.Dataset(
                {'tas': (('time', 'lat', 'lon'), field.astype('float32'),
                         {'units': 'K', 'long_name': 'near-surface air temperature'})},
                coords={'time': time, 'lat': lat.astype('float32'), 'lon': lon.astype('float32')},
                attrs={'source_id': model, 'experiment_id': exp, 'note': 'SYNTHETIC — not real CMIP6 data'},
            )
            out[f'{model}.{exp}'] = ds
    return out

USE_PANGEO = HAVE_INTAKE  # set to False to force the offline path

if not USE_PANGEO:
    print('Using SYNTHETIC fallback data — fine for developing the pipeline.')
    raw = make_synthetic_cmip6()
    for k, v in raw.items():
        print(f'  {k}: time={v.dims["time"]}, lat={v.dims["lat"]}, lon={v.dims["lon"]}')

In [ ]:
# TODO Stage 1 — Pangeo CMIP6 path
# Prompt your AI to fill this in. Hints below.
#
# 1. col = intake.open_esm_datastore('https://storage.googleapis.com/cmip6/pangeo-cmip6.json')
# 2. cat = col.search(variable_id=VARIABLE, experiment_id=['historical', *SCENARIOS],
#                     table_id='Amon', source_id=['MPI-ESM1-2-LR','CanESM5','MIROC6'],
#                     member_id='r1i1p1f1')
# 3. raw = cat.to_dataset_dict(zarr_kwargs={'consolidated': True}, storage_options={'token':'anon'})
#
# After running, inspect:
#   - list(raw.keys())  -> are all (model, scenario) pairs present?
#   - raw[<a key>]      -> what coords does it have? Is time CF-compliant?
#
# Common pitfalls to ask the AI about:
#   - Different models use different calendars (noleap vs 365_day). xr.merge will fail unless
#     you decode times or coerce to a common calendar.
#   - Some chunked zarrs need .chunk({'time': 12}) before downstream ops to avoid OOM.

if USE_PANGEO:
    pass  # <-- replace with the AI-generated query
    # raw = ...

## Stage 2 — Process

**Your task:**

1. **Regrid** every model to a common 1°×1° lat/lon grid (so they're comparable).
2. **Subset** to your region of interest.
3. **Stack** the regridded models into a single Dataset with a `model` dimension.
4. Concatenate `historical` (up to 2014) with each `ssp` to make full 1850–2100 series.

**Suggested prompt:**

> *I have a dict of CMIP6 monthly tas Datasets keyed by `{model}.{experiment}`. I need to:*
> *(1) regrid each to a common 1°×1° target grid using `xesmf` conservative regridding,*
> *(2) subset to lat={lat_min}–{lat_max}, lon={lon_min}–{lon_max}, (3) concat historical*
> *with each ssp scenario along the `time` axis, and (4) stack across models into a*
> *single Dataset with a `model` dimension. Show me the resulting dimensions. Use*
> *xesmf 0.8 syntax — the API changed in recent versions.*

Trip-hazards: **longitude convention** (some models are 0–360, some are −180–180 — convert
before subsetting); **time alignment** (calendars may differ); **regridder reuse** (build it
once per source grid and reuse it — building a regridder is slow).

### Target grid

In [ ]:
# Build a common 1° lat/lon target grid covering your region (with a margin)
margin = 5.0
target_lat = np.arange(REGION['lat_min'] - margin, REGION['lat_max'] + margin + 0.01, TARGET_RES)
target_lon = np.arange(REGION['lon_min'] - margin, REGION['lon_max'] + margin + 0.01, TARGET_RES)

target_grid = xr.Dataset(
    coords={
        'lat': ('lat', target_lat, {'units': 'degrees_north'}),
        'lon': ('lon', target_lon, {'units': 'degrees_east'}),
    }
)
print(f'Target grid: {target_grid.sizes["lat"]} lat × {target_grid.sizes["lon"]} lon')

In [ ]:
# TODO Stage 2 — regrid + subset + concat + stack
# Prompt your AI to fill this in. A scaffolding sketch:
#
# def to_180(ds):
#     '''Convert 0..360 longitude to -180..180 if needed.'''
#     if ds.lon.max() > 180:
#         ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby('lon')
#     return ds
#
# regridded = {}
# for key, ds in raw.items():
#     ds = to_180(ds)
#     # Build regridder (xesmf) and apply...
#     # Subset to region...
#     regridded[key] = ...
#
# Then for each scenario:
#   - Concatenate hist + ssp along time
#   - Stack models along a new 'model' dimension
# End up with two Datasets: combined_ssp245, combined_ssp585
# Each has dims: (model, time, lat, lon)

# REPLACE the next line with your real implementation
combined_ssp245 = None
combined_ssp585 = None

## Stage 3 — Analyze

**Your task:**

1. Compute the **reference climatology** (1995–2014 mean) per model.
2. Compute the **future mean** (2081–2100) per model per scenario.
3. **Warming = future − reference**, per model, per grid cell.
4. **Multi-model mean** and **multi-model spread** (std across the model dim).
5. Compute a **model-agreement mask**: fraction of models that agree on sign of change
   (use this for stippling in the figure).

**Suggested prompt:**

> *Given an xarray Dataset `combined` with dims (model, time, lat, lon) holding tas in K,*
> *and a reference period {ref_start}–{ref_end}, compute per-model warming as*
> *(2081–2100 mean) − (reference mean). Then compute the multi-model mean and a*
> *robust-change mask: True where >= 80% of models agree on the sign of warming.*
> *Return a Dataset with vars `warming_mean`, `warming_std`, `agreement`.*
> *Use xarray 2024 syntax.*

In [ ]:
# TODO Stage 3 — compute warming statistics per scenario
#
# def warming_stats(combined, ref=HISTORICAL_REF, fut=FUTURE_PERIOD, threshold=0.8):
#     ref_mean = combined.sel(time=slice(*ref)).mean('time')
#     fut_mean = combined.sel(time=slice(*fut)).mean('time')
#     warming = fut_mean - ref_mean         # dims (model, lat, lon)
#     mean = warming.mean('model')
#     std  = warming.std('model')
#     n_pos = (warming > 0).sum('model')
#     agreement = (n_pos / combined.sizes['model']) >= threshold
#     return xr.Dataset({'warming_mean': mean, 'warming_std': std, 'agreement': agreement})
#
# stats_245 = warming_stats(combined_ssp245)
# stats_585 = warming_stats(combined_ssp585)

stats_245 = None
stats_585 = None

## Stage 4 — Visualize

**Your task:** make a **two-panel cartopy map** — `ssp245` on the left, `ssp585` on the right —
showing 2081–2100 minus 1995–2014 warming in your region. Add **stippling** in regions where
your `agreement` mask is True. Use a diverging colormap centered on zero (e.g., `RdBu_r`).

**Suggested prompt:**

> *Make a two-panel cartopy figure (PlateCarree projection) with coastlines and country*
> *borders showing tas warming (`warming_mean.tas`) for ssp245 and ssp585 in {region_name}.*
> *Use RdBu_r centered at 0, shared colorbar, with stippling (using `ax.contourf` with*
> *hatching like `'..'`) where `agreement` is True. Titles should show the scenario and the*
> *count of models. Keep the figure publication-clean — no chartjunk, single shared colorbar.*

Common pitfalls: forgetting `transform=ccrs.PlateCarree()` when plotting data, hatching with
the wrong levels (use `levels=[0, 0.5, 1]` and hatch only the True region), and putting
stippling **everywhere** if your agreement threshold is too low.

In [ ]:
# TODO Stage 4 — two-panel cartopy figure with stippling
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 5),
#                          subplot_kw={'projection': ccrs.PlateCarree()})
# vmax = max(stats_245.warming_mean.tas.max(), stats_585.warming_mean.tas.max())
# for ax, stats, label in zip(axes, [stats_245, stats_585], ['SSP2-4.5', 'SSP5-8.5']):
#     im = ax.pcolormesh(stats.lon, stats.lat, stats.warming_mean.tas,
#                        cmap='RdBu_r', vmin=-vmax, vmax=vmax,
#                        transform=ccrs.PlateCarree())
#     # Stippling where agreement is True
#     ax.contourf(stats.lon, stats.lat, stats.agreement.tas.astype(int),
#                 levels=[0.5, 1.5], hatches=['..'], colors='none',
#                 transform=ccrs.PlateCarree())
#     ax.coastlines()
#     ax.add_feature(cfeature.BORDERS, lw=0.4)
#     ax.set_title(f'{label}: 2081–2100 vs 1995–2014')
#
# fig.colorbar(im, ax=axes, orientation='horizontal',
#              label='ΔT (K)', shrink=0.6, pad=0.08)
# plt.show()

pass

## Reference — what a good final figure looks like

Your final figure should have:

- **Two panels** side by side, one per scenario, with the **same color scale** so they're comparable.
- **Coastlines and country borders** for geographic context.
- A **diverging colormap centered on zero** (warming > 0 is red, cooling < 0 is blue — though
  for late-21st-century projections you'll see warming basically everywhere).
- **Stippling** to show where most models agree on the sign of the change.
- A **single shared horizontal colorbar** with units (K).
- **Clear titles** identifying the scenario and the time periods being compared.

What it should *not* have: a separate colorbar per panel, missing units, mismatched color
scales between panels, and stippling everywhere (means your agreement threshold is too lenient).

## Stretch goals

1. **Area-weighted regional mean:** report the cos(lat)-weighted regional mean warming
   for each model — and the multi-model 5th/50th/95th percentile.
2. **Decadal evolution:** plot a time series of the regional-mean tas anomaly, with the
   multi-model envelope as a shaded band.
3. **Bias correction (advanced):** before computing warming, bias-correct each model's
   historical period against an observed reference (e.g., CRU TS or ERA5 climatology) using
   quantile mapping. Then redo the future projection.

## Wrap-up: write 3 sentences

In the cell below, answer:

1. What was the **one prompt** that produced the best code in this notebook? Why?
2. What was the **one prompt** that produced clearly wrong code? What gave it away?
3. What's **one verification check** you did manually that the AI couldn't have done for you?

In [ ]:
# YOUR WRITE-UP
# 1. Best prompt: ...
# 2. Worst prompt: ...
# 3. Manual verification: ...
